# Comparing all the models

All five models side by side. Reads whatever `results/preds_*_test.csv` files exist, so it works now and I just rerun it as the rest land. Nothing gets retrained.

The last part is the one that matters. Everyone scores around 0.98, so the real question is who still works on tweets with no trigger word in them.

In [1]:
!git clone -q https://github.com/Ebi-Tech/NLP-and-Language-Technologies.git
%cd NLP-and-Language-Technologies

/content/NLP-and-Language-Technologies


In [2]:
import re
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_curve, roc_auc_score

from src import metrics as M
from src.save_preds import save_preds
from src.robustness import top_keywords

PROB_COLS = [f"p_{label}" for label in M.LABEL_ORDER]

## Step 1: Load the predictions

Everyone saved through `src/save_preds.py` so the columns match. I sort by `Tweet_ID` so the rows line up.

In [3]:
preds = {}
for path in sorted(glob.glob("reports/results/preds_*_test.csv")):
    name = Path(path).stem.replace("preds_", "").replace("_test", "")
    preds[name] = pd.read_csv(path).sort_values("Tweet_ID").reset_index(drop=True)

print("found:", list(preds))
for name, d in preds.items():
    print(f"  {name}: {len(d)} rows")

found: ['bilstm', 'distilbert', 'majority', 'textcnn', 'tfidf_logreg']
  bilstm: 5948 rows
  distilbert: 5948 rows
  majority: 5948 rows
  textcnn: 5948 rows
  tfidf_logreg: 5948 rows


## Step 2: Add the majority baseline

Always predicts the biggest class. Nothing to train, so I just build it here. It's the floor: ~82% accuracy from imbalance alone.

In [4]:
if "majority" not in preds and preds:
    ref = next(iter(preds.values()))
    probs = np.zeros((len(ref), len(M.LABEL_ORDER)))
    probs[:, 4] = 1.0
    out = save_preds("majority", ref["Tweet_ID"], probs, ref["true_id"])
    preds["majority"] = pd.read_csv(out).sort_values("Tweet_ID").reset_index(drop=True)
    print("made", out)

ids = [tuple(d["Tweet_ID"]) for d in preds.values()]
truth = [tuple(d["true_id"]) for d in preds.values()]
print("same tweets in every file:", len(set(ids)) == 1)
print("same labels in every file:", len(set(truth)) == 1)

same tweets in every file: True
same labels in every file: True


## Step 3: The table

All computed from the predictions with the shared `compute_metrics`, so no model is scored by different code.

Read macro F1, not accuracy.

In [5]:
rows = []
for name, d in preds.items():
    m = M.compute_metrics(d["true_id"], d["pred_id"])
    row = {"model": name, "accuracy": m["accuracy"], "macro_f1": m["macro_f1"]}
    for label in M.LABEL_ORDER:
        row[f"f1_{label}"] = m["per_class_report"][label]["f1-score"]
    rows.append(row)

table = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
table.round(4)

,model,accuracy,macro_f1,f1_economic_violence,f1_emotional_violence,f1_harmful_traditional_practice,f1_physical_violence,f1_sexual_violence
0,textcnn,0.9995,0.9978,1.0000,0.9897,1.0000,0.9994,0.9997
1,distilbert,0.9993,0.9954,0.9841,0.9949,1.0000,0.9983,0.9997
2,tfidf_logreg,0.9970,0.9813,0.9697,0.9608,0.9825,0.9955,0.9983
3,bilstm,0.9960,0.9693,0.9412,0.9282,0.9825,0.9972,0.9976
4,majority,0.8235,0.1806,0.0000,0.0000,0.0000,0.0000,0.9032


## Step 4: Confusion matrices

One grid, all models. I draw my own so I don't overwrite Jean's Part 1 figures.

In [6]:
SHORT = ["economic", "emotional", "harmful trad.", "physical", "sexual"]

show = [(n, d) for n, d in preds.items() if n != "majority"]
ncol = 2
nrow = int(np.ceil(len(show) / ncol))

fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 4.6 * nrow))
axes = np.atleast_1d(axes).ravel()

for ax, (name, d) in zip(axes, show):
    m = M.compute_metrics(d["true_id"], d["pred_id"])
    cm = m["confusion_matrix"]
    ax.imshow(cm, cmap="Blues")
    ax.set_title(f"{name}  (macro F1 {m['macro_f1']:.4f})", fontsize=10)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(SHORT, rotation=35, ha="right", fontsize=8)
    ax.set_yticklabels(SHORT, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    for r in range(5):
        for c in range(5):
            ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=8,
                    color="white" if cm[r, c] > cm.max() * 0.6 else "black")

for ax in axes[len(show):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("reports/figures/comparison_confusion_matrices.png", dpi=150)
plt.show()

C:\Users\jeanm\AppData\Local\Temp\ipykernel_5868\3271774616.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: ROC curves

Needs the probabilities, not the labels. 5 classes so it's one against the rest, averaged into one line per model.

Averaged across classes, not tweets, otherwise `sexual_violence` drowns everything out.

In [7]:
grid = np.linspace(0, 1, 2000)

def macro_roc(y, P):
    tprs = []
    for i in range(len(M.LABEL_ORDER)):
        fpr, tpr, _ = roc_curve((y == i).astype(int), P[:, i])
        tprs.append(np.interp(grid, fpr, tpr))
    return np.mean(tprs, axis=0)

auc_rows = []
fig, (full, zoom) = plt.subplots(1, 2, figsize=(11, 4.6))

for name, d in preds.items():
    y = d["true_id"].values
    P = d[PROB_COLS].values
    curve = macro_roc(y, P)
    full.plot(grid, curve, label=name)
    zoom.plot(grid, curve, marker="", label=name)

    row = {"model": name}
    for i, label in enumerate(M.LABEL_ORDER):
        row[f"auc_{label}"] = roc_auc_score((y == i).astype(int), P[:, i])
    row["macro_auc"] = np.mean([row[f"auc_{l}"] for l in M.LABEL_ORDER])
    auc_rows.append(row)

full.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
full.set_title("Macro averaged ROC")
zoom.set_xlim(0, 0.02)
zoom.set_ylim(0.9, 1.005)
zoom.set_title("Top left corner, zoomed")

for ax in (full, zoom):
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("reports/figures/comparison_roc.png", dpi=150)
plt.show()

auc = pd.DataFrame(auc_rows).sort_values("macro_auc", ascending=False).reset_index(drop=True)
auc.round(5)

C:\Users\jeanm\AppData\Local\Temp\ipykernel_5868\214302604.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,auc_economic_violence,auc_emotional_violence,auc_harmful_traditional_practice,auc_physical_violence,auc_sexual_violence,macro_auc
0,textcnn,1.00000,0.99995,1.0,1.00000,0.99999,0.99999
1,tfidf_logreg,1.00000,0.99998,1.0,0.99994,0.99990,0.99997
2,distilbert,0.99927,1.00000,1.0,1.00000,0.99998,0.99985
3,bilstm,0.99992,0.99581,1.0,0.99997,0.99904,0.99895
4,majority,0.50000,0.50000,0.5,0.50000,0.50000,0.50000


## Step 6: Stress tests

I dropped the earlier idea of filtering out tweets that contain a keyword. Almost every tweet contains one, so the subset came out as a handful of tweets and told us nothing.

Instead each model runs the tests from `src/robustness.py` on its own and saves the result. This reads those files. Blank cells mean that model hasn't run them yet.

- **masked**: the strongest words per class replaced
- **random masked**: the same number of random words replaced, as the control
- **shuffled**: word order destroyed

In [8]:
import json as _json

VARIANTS = ["masked", "random_masked", "shuffled"]

stress = []
for name, d in preds.items():
    row = {"model": name, "normal": M.compute_metrics(d["true_id"], d["pred_id"])["macro_f1"]}
    for v in VARIANTS:
        f = Path(f"reports/results/{name}_{v}_test.json")
        row[v] = _json.load(open(f))["macro_f1"] if f.exists() else None
    stress.append(row)

stress = pd.DataFrame(stress)
stress.round(4)

,model,normal,masked,random_masked,shuffled
0,bilstm,0.9693,0.1993,0.7453,0.9295
1,distilbert,0.9954,0.2463,0.9207,0.9857
2,majority,0.1806,NaN,NaN,NaN
3,textcnn,0.9978,0.3999,0.8914,0.9978
4,tfidf_logreg,0.9813,0.2485,0.9351,0.9777


In [9]:
plot_me = stress.dropna(subset=VARIANTS, how="all").reset_index(drop=True)
cols = ["normal"] + VARIANTS
x = np.arange(len(plot_me))
width = 0.8 / len(cols)

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for k, col in enumerate(cols):
    off = (k - (len(cols) - 1) / 2) * width
    vals = plot_me[col]
    bars = ax.bar(x + off, vals.fillna(0), width, label=col.replace("_", " "))
    for b, v in zip(bars, vals):
        if pd.notna(v):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f"{v:.2f}",
                    ha="center", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(plot_me["model"])
ax.set_ylabel("macro F1")
ax.set_ylim(0, 1.12)
ax.set_title("What survives the stress tests")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig("reports/figures/comparison_stress_tests.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\jeanm\AppData\Local\Temp\ipykernel_5868\3028720317.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7: What's being missed

The tweets each model gets wrong, for the error analysis.

In [10]:
text = pd.read_csv("data/processed/train_cleaned.csv")
lookup = text.set_index("Tweet_ID")["tweet_clean"]

for name, d in preds.items():
    wrong = d[d["true_id"] != d["pred_id"]]
    print(f"\n=== {name}: {len(wrong)} wrong out of {len(d)} ===")
    for _, r in wrong.head(3).iterrows():
        print(f"  {M.LABEL_ORDER[r['true_id']]} -> said {M.LABEL_ORDER[r['pred_id']]}")
        print(f"  {lookup[r['Tweet_ID']][:200]}")


=== bilstm: 24 wrong out of 5948 ===
  physical_violence -> said economic_violence
  hi ya'll! our lives suck! all we gots is jesus, the flag and donnnnald trump! my husband 1) left me; 2) beats me; 3) ignores me. i have no life. i have a shitty job and i don't even know my kids. lock
  emotional_violence -> said sexual_violence
  i've just been verbally abused on the bus. a young, male, british drunk swore at another passenger in front of kids. asked him to mind his language because daytime, public transport, rude &amp; kids. 
  sexual_violence -> said physical_violence
  calling for my beating at home from my husband, calling for me to be raped if i don't dress the part, calling for the segregation between muslims and coptic christians of the same blood, calling for c

=== distilbert: 4 wrong out of 5948 ===
  sexual_violence -> said physical_violence
  u may be right. but if she claims her husband beats her, indulges in un natural sex, had raped their minor daughter, his father rap

In [11]:
table.to_csv("reports/results/model_comparison_all.csv", index=False)
auc.to_csv("reports/results/roc_auc_all.csv", index=False)
stress.to_csv("reports/results/stress_test_comparison.csv", index=False)
print("saved 3 tables and 3 figures")

saved 3 tables and 3 figures
